# OmniVoice Fast Inference (FP16 + FlashInfer)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fauziardiantama/aichan-extension/blob/main/OmniVoice.ipynb)

Notebook ini menjalankan inferensi cepat OmniVoice menggunakan:
1. **Model Lokal FP16** dari Google Drive (`aichan-extension-files/omnivoice-float16`).
2. **Akselerasi FlashInfer** (`apply_flashinfer` dengan CUDA Graph).
3. **Pre-computed Prompt** (`aichan-extension-files/sample.pt`), tanpa perlu memuat Whisper ASR atau audio raw.
4. **Optimasi Native Runtime**: `use_cache=False`, `torch.inference_mode()`, dan `num_step=16` (2x lebih cepat).

## 1. Install Dependencies

In [ ]:
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git flashinfer-python

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Load Local FP16 Model, Apply FlashInfer & Load Prompt

In [ ]:
import os
import torch
from omnivoice import OmniVoice, VoiceClonePrompt
from omnivoice.models.omnivoice_flashinfer import apply_flashinfer

base_dir = "/content/drive/MyDrive/aichan-extension-files"
model_path = os.path.join(base_dir, "omnivoice-float16")
prompt_path = os.path.join(base_dir, "sample.pt")

# 1. Muat checkpoint lokal FP16 tanpa Whisper ASR
print(f"Memuat model lokal FP16 dari: {model_path}...")
model = OmniVoice.from_pretrained(
    model_path,
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=False,
)

# 2. Matikan KV cache Transformer (menghilangkan overhead DynamicCache)
model.llm.config.use_cache = False

# 3. Pasang akselerasi FlashInfer dengan CUDA Graph
print("Mengaktifkan akselerasi FlashInfer...")
apply_flashinfer(model, enable_cuda_graph=True)

# 4. Muat pre-computed voice clone prompt
print(f"Memuat prompt suara dari: {prompt_path}...")
prompt = VoiceClonePrompt.load(prompt_path)

print("\nModel dan prompt siap digunakan!")

## 4. Generate Audio (Fast Native Inference)

In [ ]:
# @title Input Teks & Generate Audio
text = "Halo, ini adalah pengujian sintesis suara OmniVoice dengan model lokal FP16 dan akselerasi FlashInfer."  # @param {type:"string"}

import soundfile as sf
from IPython.display import Audio, display

print("Menghasilkan audio...")
with torch.inference_mode():
    audio = model.generate(
        text=text,
        voice_clone_prompt=prompt,
        num_step=16,
    )

output_file = "output.wav"
sf.write(output_file, audio[0], 24000)
print(f"Audio tersimpan di: {output_file}")

display(Audio(audio[0], rate=24000))